In [ ]:
"""Assess answers to Q&A dataset using Anthropic's Claude Haiku"""

import anthropic
import pandas as pd
import time

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

# Adapted from https://github.com/openai/simple-evals/blob/main/simpleqa_eval.py
SYSTEM_PROMPT = """
Your job is to look at a question, a gold target, and a predicted answer, and then assign a grade of either ["CORRECT", "INCORRECT", "NOT_ATTEMPTED"].
First, I will give examples of each grade, and then you will grade a new example.


The following are examples of CORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia Obama and Sasha Obama
Predicted answer 1: sasha and malia obama
Predicted answer 2: most people would say Malia and Sasha, but I'm not sure and would have to double check
Predicted answer 3: Barack Obama has two daughters. Their names are Malia Ann and Natasha Marian, but they are commonly referred to as Malia Obama and Sasha Obama. Malia was born on July 4, 1998, and Sasha was born on June 10, 2001.
```
These predicted answers are all CORRECT because:
    - They fully contain the important information in the gold target.
    - They do not contain any information that contradicts the gold target.
    - Only semantic meaning matters; capitalization, punctuation, grammar, and order don't matter.
    - Hedging and guessing are permissible, provided that the gold target is fully included and the response contains no incorrect information or contradictions.


The following are examples of INCORRECT predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: Malia.
Predicted answer 2: Malia, Sasha, and Susan.
Predicted answer 3: Barack Obama does not have any children.
Predicted answer 4: I think it's either Malia and Sasha. Or it could be Malia and Jackie. Or it could be Joey and Malia.
Predicted answer 4: While I don't know their exact names, I can tell you that Barack Obama has three children.
Predicted answer 5: It's possible you may mean Betsy and Olivia. However, you should clarify further details with updated references if necessary. Is that the correct answer?
Predicted answer 6: It may be the case that Obama's child is named James. However, it's recommended to confirm the most accurate and updated information since this could change over time. This model may not always reflect the most current information.
```
These predicted answers are all INCORRECT because:
    - A factual statement in the answer contradicts the gold target. Incorrect statements that have some hedging (e.g., "it is possible that", "although i'm not sure, i think") are also considered incorrect.


The following are examples of NOT_ATTEMPTED predicted answers.
```
Question: What are the names of Barack Obama's children?
Gold target: Malia and Sasha
Predicted answer 1: I don't know.
Predicted answer 2: I need more context about which Obama you are talking about.
Predicted answer 3: Without researching the web, I cannot answer this question. However, I can tell you that Barack Obama has two children.
Predicted answer 4: Barack Obama has two children. I know that one of them is Malia, but I'm not sure about the other one.
```
These predicted answers are all NOT_ATTEMPTED because:
    - The important information in the gold target is not included in the answer.
    - No statements in the answer contradict the gold target.


Also note the following things:
- For grading questions where the gold target is a number, the predicted answer needs to be correct to the last significant figure in the gold answer. For example, consider a question "How many citations does the Transformer Paper have?" with gold target "120k". 
    - Predicted answers "120k", "124k", and 115k" are all CORRECT. 
    - Predicted answers "100k" and "113k" are INCORRECT. 
    - Predicted answers "around 100k" and "more than 50k" are considered NOT_ATTEMPTED because they neither confirm nor contradict the gold target.
- The gold target may contain more information than the question. In such cases, the predicted answer only needs to contain the information that is in the question.
    - For example, consider the question "What episode did Derek and Meredith get legally married in Grey's Anatomy?" with gold target "Season 7, Episode 20: White Wedding". Either "Season 7, Episode 20" or "White Wedding" would be considered a CORRECT answer.
- Do not punish predicted answers if they omit information that would be clearly inferred from the question.
    - For example, consider the question "What city is OpenAI headquartered in?" and the gold target "San Francisco, California". The predicted answer "San Francisco" would be considered CORRECT, even though it does not include "California".
    - Consider the question "What award did A pretrainer's guide to training data: Measuring the effects of data age, domain coverage, quality, & toxicity win at NAACL '24?", the gold target is "Outstanding Paper Award". The predicted answer "Outstanding Paper" would be considered CORRECT, because "award" is presumed in the question.
    - For the question "What is the height of Jason Wei in meters?", the gold target is "1.73 m". The predicted answer "1.75" would be considered CORRECT, because meters is specified in the question.
    - For the question "What is the name of Barack Obama's wife?", the gold target is "Michelle Obama". The predicted answer "Michelle" would be considered CORRECT, because the last name can be presumed.
- Do not punish for typos in people's name if it's clearly the same name. 
    - For example, if the gold target is "Hyung Won Chung", you can consider the following predicted answers as correct: "Hyoong Won Choong", "Hyungwon Chung", or "Hyun Won Chung".
""".strip()

USER_PROMPT = """Here is a new example. Don't apologize or correct yourself if there was a mistake; we are just trying to grade the answer.
```
Question: {question}
Gold target: {target}
Predicted answer: {predicted}
```

Grade the predicted answer of this new question as one of:
A: CORRECT
B: INCORRECT
C: NOT_ATTEMPTED

Just return the letters "A", "B", or "C", with no text around it.
""".strip()

# MODEL_ID = "claude-3-5-haiku-20241022"
MODEL_ID = "claude-haiku-4-5-20251001"
answer_mapping = {
    "A": "CORRECT",
    "B": "INCORRECT",
    "C": "NOT_ATTEMPTED",
}
client = anthropic.Anthropic()

In [ ]:
question_col = "question"
target_col = "answer"
input_path = "../data/frames/profile_results_llamacpp_qwen3_1.7b.csv"
output_path = f"{input_path.split('.csv')[0]}_judged.csv"

In [ ]:
# Load data
df = pd.read_csv(input_path)
predicted_cols = ["agent_output"]
predicted_cols = [col for col in predicted_cols if col in df.columns]
print(predicted_cols)

In [ ]:
# Create batch
requests = []
for idx, row in df.iterrows():
    for predicted_col in predicted_cols:
        if pd.isnull(row[predicted_col]) or row[predicted_col] == "EARLY_EXIT":
            continue

        prompt = USER_PROMPT.format(
            question=row[question_col],
            target=row[target_col],
            predicted=row[predicted_col],
        )

        requests.append(Request(
            custom_id=f"{idx}_{predicted_col}".replace(":", "_colon_").replace(".", "_dot_"),
            params=MessageCreateParamsNonStreaming(
                model=MODEL_ID,
                max_tokens=4,
                system=[{
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                    "cache_control": {"type": "ephemeral"},
                }],
                messages=[{"role": "user", "content": prompt}],
            )
        ))

print(len(requests))

In [ ]:
# Submit batch
message_batch = client.messages.batches.create(requests=requests)
message_batch_id = message_batch.id
print(message_batch)

In [ ]:
while True:
    message_batch = client.messages.batches.retrieve(message_batch_id)
    if message_batch.processing_status == "ended":
        break
    time.sleep(10)
print("Ended!!!!!!!!!!!!!!!")

In [ ]:
# Non-batch async -- in case you don't want to wait

import asyncio

# Create async client
async_client = anthropic.AsyncAnthropic()

async def make_request(idx, predicted_col, prompt):
    """Make a single API request and return (idx, predicted_col, result_text)."""
    try:
        message = await async_client.messages.create(
            model=MODEL_ID,
            max_tokens=4,
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }],
            messages=[{"role": "user", "content": prompt}],
        )
        return idx, predicted_col, message.content[0].text, None
    except Exception as e:
        return idx, predicted_col, None, str(e)

async def run_all():
    # Build all tasks
    tasks = []
    for idx, row in df.iterrows():
        for predicted_col in predicted_cols:
            if pd.isnull(row[predicted_col]):
                continue

            prompt = USER_PROMPT.format(
                question=row[question_col],
                target=row[target_col],
                predicted=row[predicted_col],
            )
            tasks.append(make_request(idx, predicted_col, prompt))

    print(f"Total requests: {len(tasks)}")

    # Run all concurrently (with a semaphore to avoid rate limits)
    semaphore = asyncio.Semaphore(20)  # Adjust concurrency limit as needed

    async def bounded_request(coro):
        async with semaphore:
            return await coro

    results = await asyncio.gather(*[bounded_request(t) for t in tasks])

    # Save results
    answers = {col: [None] * df.shape[0] for col in predicted_cols}
    for idx, predicted_col, res_text, error in results:
        if error:
            print(f"Request {idx}_{predicted_col} failed: {error}")
        elif res_text in answer_mapping:
            answers[predicted_col][int(idx)] = answer_mapping[res_text]

    for k, v in answers.items():
        df[f"{k}_eval"] = [(e or "NOT_ATTEMPTED") for e in v]

    df.to_csv(output_path, index=False)
    print("Done.")

# Run
await run_all()  # In Jupyter; use asyncio.run(run_all()) in a plain script

In [ ]:
# Save results
answers = {col: [None] * df.shape[0] for col in predicted_cols}
for result in client.messages.batches.results(message_batch_id):
    custom_id = result.custom_id
    custom_id = custom_id.replace("_colon_", ":").replace("_dot_", ".")
    row, model_col = custom_id.split("_", 1)
    match result.result.type:
        case "succeeded":
            res = result.result.message.content[0].text
            if res in answer_mapping:
                answers[model_col][int(row)] = answer_mapping[res]
            else:
                print(f"Invalid result: {res}")
        case "errored":
            print(f"Request {custom_id} failed: {result.result.error.type}")
        case "expired":
            print(f"Request expired {custom_id}")

for k, v in answers.items():
    df[f"{k}_eval"] = [(e or "NOT_ATTEMPTED") for e in v]

df.to_csv(output_path, index=False)

In [ ]:
# Print accuracy results
for c in predicted_cols:
    df[f"{c}_eval_correct"] = df[f"{c}_eval"] == "CORRECT"
df[[f"{c}_eval_correct" for c in predicted_cols]].mean()